In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir(r"E:\retailpulse")

df = pd.read_csv("data/processed/master.csv",
                 parse_dates=["order_date"])

# ── ABC Analysis ───────────────────────────────────────────
# A = top 80% revenue products
# B = next 15% revenue products  
# C = bottom 5% revenue products

product_rev = (df.groupby(["product_id","category"])
                 .agg(
                     total_revenue = ("revenue",      "sum"),
                     total_units   = ("order_item_id","count"),
                     avg_price     = ("price",        "mean"),
                     num_orders    = ("order_id",     "nunique")
                 )
                 .reset_index()
                 .sort_values("total_revenue", ascending=False))

# Cumulative revenue percentage
product_rev["cum_pct"] = (product_rev["total_revenue"].cumsum() /
                           product_rev["total_revenue"].sum() * 100)

# ABC classification
def abc_class(cum_pct):
    if cum_pct <= 80:
        return "A"
    elif cum_pct <= 95:
        return "B"
    else:
        return "C"

product_rev["abc_class"] = product_rev["cum_pct"].apply(abc_class)

print("📊 ABC Analysis Results:")
abc_summary = product_rev.groupby("abc_class").agg(
    products      = ("product_id",    "count"),
    total_revenue = ("total_revenue", "sum"),
    avg_price     = ("avg_price",     "mean")
).round(2).reset_index()

abc_summary["revenue_pct"] = (abc_summary["total_revenue"] /
                               abc_summary["total_revenue"].sum()
                               * 100).round(1)
abc_summary["product_pct"] = (abc_summary["products"] /
                               len(product_rev) * 100).round(1)

print(abc_summary.to_string(index=False))

📊 ABC Analysis Results:
abc_class  products  total_revenue  avg_price  revenue_pct  product_pct
        A      8911    12171918.94     308.59         80.0         28.2
        B     11291     2282267.85     117.60         15.0         35.7
        C     11417      760854.20      44.06          5.0         36.1


In [2]:
# Economic Order Quantity (EOQ)
# EOQ = sqrt(2 * D * S / H)
# D = annual demand (units)
# S = ordering cost (fixed, assumed BRL 50)
# H = holding cost per unit per year (assumed 20% of price)

ORDERING_COST = 50   # BRL per order
HOLDING_RATE  = 0.20 # 20% of unit price per year

# Calculate annual demand
# Dataset covers ~2 years so divide by 2
product_rev["annual_demand"] = product_rev["total_units"] / 2

# Holding cost per unit
product_rev["holding_cost"] = product_rev["avg_price"] * HOLDING_RATE

# EOQ formula
product_rev["EOQ"] = np.sqrt(
    2 * product_rev["annual_demand"] * ORDERING_COST /
    (product_rev["holding_cost"] + 1e-6)).round(0).astype(int)

# Reorder frequency (times per year)
product_rev["reorder_frequency"] = (
    product_rev["annual_demand"] /
    (product_rev["EOQ"] + 1e-6)).round(1)

# Safety stock (1 week of average demand)
product_rev["weekly_demand"] = product_rev["annual_demand"] / 52
product_rev["safety_stock"]  = (product_rev["weekly_demand"] * 1.5).round(0).astype(int)

print("📦 EOQ Results — Top 10 Class A Products:")
top_a = product_rev[product_rev["abc_class"]=="A"].head(10)
print(top_a[["product_id","category","abc_class",
             "annual_demand","EOQ","safety_stock",
             "reorder_frequency"]].to_string(index=False))

📦 EOQ Results — Top 10 Class A Products:
                      product_id              category abc_class  annual_demand  EOQ  safety_stock  reorder_frequency
bb50f2e236e5eea0100680137654686c         health_beauty         A           97.0   12             3                8.1
d1c427060a0f73f6b889a5c7c61f2ac4 computers_accessories         A          166.0   25             5                6.6
6cdd53843498f92890544667809f1595         health_beauty         A           76.5   10             2                7.6
99a4788cb24856965c36a24e339b6058        bed_bath_table         A          238.5   37             7                6.4
3dd2a17168ec895c781a9191c1e95ad7 computers_accessories         A          136.0   21             4                6.5
d6160fb7873f184099d9bc95e30376af             computers         A           16.5    2             0                8.2
aca2eb7d00ea1a7b8ebd4e68314663af       furniture_decor         A          260.0   43             8                6.0
5f504b3a1c75b73

In [3]:
# ABC pie chart
abc_plot = product_rev.groupby("abc_class")["total_revenue"].sum().reset_index()

fig = px.pie(abc_plot,
             values="total_revenue",
             names="abc_class",
             title="ABC Analysis — Revenue by Class",
             color="abc_class",
             color_discrete_map={"A":"#22c55e",
                                 "B":"#f59e0b",
                                 "C":"#ef4444"})
fig.show()

# Category ABC distribution
cat_abc = (product_rev.groupby(["category","abc_class"])
                      ["product_id"].count()
                      .reset_index()
                      .rename(columns={"product_id":"count"}))

top_cats = (product_rev.groupby("category")["total_revenue"]
                       .sum()
                       .nlargest(10)
                       .index.tolist())

fig2 = px.bar(cat_abc[cat_abc["category"].isin(top_cats)],
              x="category", y="count",
              color="abc_class",
              title="ABC Classification by Category (Top 10)",
              color_discrete_map={"A":"#22c55e",
                                  "B":"#f59e0b",
                                  "C":"#ef4444"},
              barmode="stack")
fig2.update_xaxes(tickangle=45)
fig2.show()

# Summary stats
print("\n📊 Inventory Optimization Summary:")
print(f"  Total products analyzed: {len(product_rev)}")
print(f"  Class A products: "
      f"{(product_rev['abc_class']=='A').sum()} "
      f"({abc_summary[abc_summary['abc_class']=='A']['product_pct'].values[0]}%)"
      f" → drive 80% revenue")
print(f"  Avg EOQ (Class A): "
      f"{product_rev[product_rev['abc_class']=='A']['EOQ'].mean():.0f} units")
print(f"  Avg reorder frequency (Class A): "
      f"{product_rev[product_rev['abc_class']=='A']['reorder_frequency'].mean():.1f}x/year")

# Save
product_rev.to_csv("data/processed/inventory_abc.csv", index=False)
abc_summary.to_csv("data/processed/abc_summary.csv", index=False)
print("\n✅ Inventory optimization saved!")


📊 Inventory Optimization Summary:
  Total products analyzed: 31619
  Class A products: 8911 (28.2%) → drive 80% revenue
  Avg EOQ (Class A): 4 units
  Avg reorder frequency (Class A): 18461.2x/year

✅ Inventory optimization saved!
